In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
repo_root = next(
    (
        path for path in (current, *current.parents)
        if (path / "transformations" / "silver").is_dir()
        and (path / "sources" / "reference").is_dir()
    ),
    None,
)

if repo_root is None:
    raise RuntimeError("No se encontró la raíz del repositorio.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from transformations.silver.products_cdc_transform import (
    products_to_dataframe,
    apply_product_batch,
)

In [ ]:
import json

ingestion_run_id = "9625ead0-e913-5b83-873e-27b1ccda74de"

bronze_root = (
    "abfss://bronze@starlpdevdiego01.dfs.core.windows.net/"
    "sqlserver/techretail_oltp/cdc/products/"
    f"run_id={ingestion_run_id}"
)

target_path = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net/"
    "techretail/products"
)

checkpoint_path = (
    "abfss://silver@starlpdevdiego01.dfs.core.windows.net/"
    "_control/products_cdc_application"
)

table_name = "dbw_arlp_dev_diego01.silver_techretail.products"

print("ENTRADA:", bronze_root)
print("DESTINO:", target_path)
print("CONTROL:", checkpoint_path)

In [ ]:
def read_small_json(path):
    limit = 1024 * 1024
    files = dbutils.fs.ls(path)

    if len(files) != 1 or files[0].size > limit:
        raise ValueError(f"Se esperaba un archivo JSON de hasta 1 MiB: {path}")

    return json.loads(dbutils.fs.head(path, limit))


spark.conf.set("spark.sql.session.timeZone", "UTC")

baseline = read_small_json(f"{bronze_root}/baseline.json")
first_batch = read_small_json(f"{bronze_root}/batch_001.json")
second_batch = read_small_json(f"{bronze_root}/batch_002.json")

for document in (baseline, first_batch, second_batch):
    if (
        document.get("source_database") != "TechRetail_OLTP"
        or document.get("capture_instance") != "dbo_products"
    ):
        raise ValueError("Una entrada no corresponde a CDC de productos.")

if first_batch["after_lsn"] != baseline["baseline_lsn"]:
    raise ValueError("El primer lote no continúa desde la base inicial.")

if second_batch["after_lsn"] != first_batch["to_lsn"]:
    raise ValueError("El segundo lote no continúa desde el primero.")

print("Productos de base:", len(baseline["products"]))
print("Eventos del primer lote:", len(first_batch["changes"]))
print("Eventos del segundo lote:", len(second_batch["changes"]))

In [ ]:
def verify_initial_products():
    expected = products_to_dataframe(spark, baseline["products"])
    current = spark.read.format("delta").load(target_path)

    technical_columns = [
        "source_system",
        "ingestion_run_id",
        "processing_run_id",
        "source_file",
        "processed_at",
    ]
    required_columns = expected.columns + technical_columns

    if set(current.columns) != set(required_columns):
        raise ValueError("Las columnas Silver no coinciden con lo esperado.")

    commercial = current.select(*expected.columns)

    if (
        expected.exceptAll(commercial).count() != 0
        or commercial.exceptAll(expected).count() != 0
    ):
        raise ValueError("Silver no coincide con la base inicial del CDC.")

    if expected.count() != 60:
        raise ValueError("Se esperaban 60 productos iniciales.")

    print("PASS: los 60 productos de Azure coinciden con la base CDC.")
    return current

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from delta.tables import DeltaTable
from pyspark.sql import types as T


control_schema = T.StructType([
    T.StructField("target_path", T.StringType(), False),
    T.StructField("ingestion_run_id", T.StringType(), False),
    T.StructField("last_applied_lsn", T.StringType(), False),
    T.StructField("last_batch_sha256", T.StringType(), True),
    T.StructField("processing_run_id", T.StringType(), False),
    T.StructField("processed_at", T.StringType(), False),
])


def load_or_initialize_control():
    if not DeltaTable.isDeltaTable(spark, checkpoint_path):
        verify_initial_products()

        initial_control = {
            "target_path": target_path,
            "ingestion_run_id": ingestion_run_id,
            "last_applied_lsn": baseline["baseline_lsn"],
            "last_batch_sha256": None,
            "processing_run_id": str(uuid4()),
            "processed_at": datetime.now(timezone.utc).isoformat(),
        }

        (
            spark.createDataFrame([initial_control], schema=control_schema)
            .write.format("delta")
            .mode("error")
            .save(checkpoint_path)
        )
        print("PASS: control inicial creado después de comparar Silver.")

    rows = spark.read.format("delta").load(checkpoint_path).collect()
    if len(rows) != 1:
        raise ValueError("El control debe contener exactamente una fila.")

    control = rows[0].asDict()

    if (
        control["target_path"] != target_path
        or control["ingestion_run_id"] != ingestion_run_id
    ):
        raise ValueError("El control pertenece a otro destino o publicación.")

    return control

In [ ]:
def apply_cloud_batch(batch_name):
    if batch_name not in ("batch_001.json", "batch_002.json"):
        raise ValueError("Archivo de lote no admitido en esta prueba.")

    batch_path = f"{bronze_root}/{batch_name}"
    batch = read_small_json(batch_path)
    control = load_or_initialize_control()

    provenance = {
        "source_file": batch_path,
        "ingestion_run_id": control["ingestion_run_id"],
        "processing_run_id": control["processing_run_id"],
        "processed_at": datetime.fromisoformat(control["processed_at"]),
    }

    result = apply_product_batch(
        spark,
        batch,
        target_path,
        control,
        provenance=provenance,
    )

    if result["status"] == "already_applied":
        print(f"PASS: {batch_name} ya aplicado; sin nuevas escrituras.")
        return result

    current_control = (
        spark.read.format("delta")
        .load(checkpoint_path)
        .collect()
    )

    if (
        len(current_control) != 1
        or current_control[0].asDict() != control
    ):
        raise RuntimeError(
            "El control cambió durante la ejecución; revisar recuperación."
        )

    next_control = {
        **control,
        **result["checkpoint"],
    }

    (
        spark.createDataFrame([next_control], schema=control_schema)
        .write.format("delta")
        .mode("overwrite")
        .save(checkpoint_path)
    )

    print(f"PASS: {batch_name} aplicado y verificado.")
    print("Último LSN aplicado:", next_control["last_applied_lsn"])
    return result

In [ ]:
first_result = apply_cloud_batch("batch_001.json")
print("Estado:", first_result["status"])

In [ ]:
replay_result = apply_cloud_batch("batch_001.json")

if replay_result["status"] != "already_applied":
    raise RuntimeError("El primer lote no se reconoció como ya aplicado.")

print("PASS: repetición exacta reconocida.")

In [ ]:
current = spark.read.format("delta").load(target_path)
commercial_columns = [
    field.name
    for field in products_to_dataframe(spark, []).schema
]

expected_records = {
    product["product_id"]: dict(product)
    for product in baseline["products"]
}

ordered_events = sorted(
    first_batch["changes"],
    key=lambda event: (
        bytes.fromhex(event["commit_lsn"][2:]),
        bytes.fromhex(event["sequence_value"][2:]),
    ),
)

for event in ordered_events:
    key = event["product_id"]
    if event["operation_code"] == 1:
        expected_records.pop(key, None)
    else:
        expected_records[key] = event

expected = products_to_dataframe(spark, list(expected_records.values()))
actual = current.select(*commercial_columns)

if (
    expected.exceptAll(actual).count() != 0
    or actual.exceptAll(expected).count() != 0
):
    raise RuntimeError("Los valores comerciales finales difieren de lo esperado.")

if current.count() != 61:
    raise RuntimeError("Se esperaban 61 productos.")

control = load_or_initialize_control()
product_61 = current.filter("product_id = 61").collect()

if len(product_61) != 1:
    raise RuntimeError("El producto 61 debe aparecer exactamente una vez.")

row = product_61[0]
expected_provenance = {
    "source_system": "sqlserver",
    "ingestion_run_id": ingestion_run_id,
    "processing_run_id": control["processing_run_id"],
    "source_file": f"{bronze_root}/batch_001.json",
}

for column, value in expected_provenance.items():
    if row[column] != value:
        raise RuntimeError(f"Procedencia incorrecta: {column}")

if row["processed_at"] is None:
    raise RuntimeError("Falta processed_at.")

print("PASS: 61 productos y valores comerciales finales verificados.")
print("PASS: producto 61 con procedencia CDC y producto 62 ausente.")

current.select(
    "product_id", "sku", "list_price",
    "source_file", "processed_at",
).filter("product_id = 61").show(truncate=False)

In [ ]:
second_result = apply_cloud_batch("batch_002.json")
print("Estado del segundo lote:", second_result["status"])

if load_or_initialize_control()["last_applied_lsn"] != second_batch["to_lsn"]:
    raise RuntimeError("El control no coincide con el segundo lote.")

print("PASS: segundo lote procesado y posición de aplicación comprobada.")

In [ ]:
registered = spark.table(table_name).select(*commercial_columns)
stored = spark.read.format("delta").load(target_path).select(
    *commercial_columns
)

if (
    registered.exceptAll(stored).count() != 0
    or stored.exceptAll(registered).count() != 0
):
    raise RuntimeError("La tabla registrada difiere del destino Delta.")

print("PASS: consulta por Unity Catalog coincide con el destino Delta.")
print("Total de productos:", registered.count())